**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Kernel Methods & RKHS

The elegant middle path between linear models and neural networks — and a proud UF lineage (information-theoretic learning and kernel adaptive filtering grew up here). Kernel trick, Gaussian processes with honest error bars, and KLMS: the [adaptive filter](../Intro_Time_Series/Intro_AdFilt_APA.ipynb) gone nonlinear.

## 1. Pre-requisites

- [Hilbert Spaces](../Intro_Math/Hilbert_Spaces/Hilbert_Spaces.ipynb) — inner products, projections (an RKHS is a Hilbert space with a bonus property).
- [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S2, [Estimation Theory](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb) S3 for the GP session.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

def rbf(A, B, ell=0.5):
    """the Gaussian (RBF) kernel — similarity that decays with distance"""
    d2 = ((A[:, None, :] - B[None, :, :])**2).sum(-1)
    return np.exp(-d2 / (2 * ell**2))

---
### 🕐 Session 1 of 3 — *The Kernel Trick* (~35 min)
**Goal:** replace inner products with kernels; fit nonlinear functions with linear algebra.
**Builds on:** [Hilbert Spaces](../Intro_Math/Hilbert_Spaces/Hilbert_Spaces.ipynb). &nbsp; **Feeds into:** Session 2 (Gaussian processes).

---

## 2. Features Without Features

💡 **Intuition.** Linear methods only see inner products $x_i^T x_j$. The trick: replace every inner product with a **kernel** $k(x_i, x_j)$ — a similarity function that secretly equals an inner product in some (possibly infinite-dimensional) feature space. You get nonlinear power at linear-algebra prices, without ever visiting the feature space. The **representer theorem** seals it: the optimal function is always a weighted sum of kernels *centered on your data* — $f(x) = \sum_i \alpha_i k(x_i, x)$ — so the infinite-dimensional search collapses to solving for $n$ numbers. An RKHS is the [Hilbert space](../Intro_Math/Hilbert_Spaces/Hilbert_Spaces.ipynb) where evaluation *is* an inner product with a kernel bump.

In [ ]:
# Kernel ridge regression from scratch: (K + λI)α = y — one linear solve, nonlinear fit

# YOUR CODE HERE


**What just happened.** A curved, kinked function fitted by **one call to `np.linalg.solve`**. No iteration, no gradient descent, no learning rate, no epochs — build the $60\times60$ kernel matrix, solve $(K + \lambda I)\alpha = y$, and evaluate. Three lines, and the result tracks a target that no straight line could approach.

**Look at the shape of the code, because it is ridge regression with one substitution.** Ordinary ridge solves $(X^TX + \lambda I)w = X^Ty$ and predicts $x_*^Tw$. Here $K$ replaces $X^TX$ and $k(x_*, X)$ replaces $x_*^T$. **Every inner product was swapped for a kernel and nothing else changed** — that is the trick, stated operationally.

**And the feature space we are implicitly working in is infinite-dimensional.** The RBF kernel corresponds to a $\phi$ with infinitely many coordinates, so a naive "map the features then fit linearly" approach is not merely expensive, it is impossible. Yet the computation above is finite and small, because the representer theorem guarantees the optimum is $f(x) = \sum_i \alpha_i k(x_i, x)$ — **one bump per data point, 60 numbers**. The infinite search collapsed to a $60\times60$ solve.

**Note where the fit is honest and where it strains.** The smooth $\sin(2x)$ portion is tracked closely. The $0.5\,\mathrm{sign}(x)$ jump at the origin is **rounded off** — and it must be, because an RBF kernel encodes a prior that functions are infinitely differentiable, and no lengthscale makes a smooth kernel produce a discontinuity. This is deliberate model misspecification, planted so the calibration audit in Session 2 has something real to detect.

**The lengthscale is doing more work than the regulariser, and it is worth experimenting with here.** At $\ell = 0.5$ the bumps are narrow enough to follow the sine. Try $\ell = 0.1$: the curve interpolates every noisy point and looks like a comb. Try $\ell = 2$: the kink vanishes and the sine flattens. **One number spans underfitting to overfitting**, which makes kernel methods unusually easy to reason about and unusually sensitive to getting that number right. Session 2's marginal likelihood is the principled way to choose it.

**One caveat that becomes the central problem of Session 3.** This scales as $O(n^3)$ to solve and $O(n^2)$ to store, because $K$ is $n \times n$. At $n = 60$ that is instant; at $n = 10^5$ the matrix alone is 80 GB. **Kernel methods are exact and elegant and they do not scale**, and that single fact — not any deficiency of accuracy — is most of why neural networks displaced them once datasets grew.

**The lengthscale is the model.** $\ell$ controls how far influence spreads — small $\ell$ wiggles (overfits), large $\ell$ oversmooths. It's the bias-variance dial in one number, and choosing it honestly (cross-validation, or S2's marginal likelihood) is most of kernel practice.

---
### 🕐 Session 2 of 3 — *Gaussian Processes* (~40 min)
**Goal:** put a prior on functions; get predictions WITH calibrated uncertainty.
**Builds on:** Session 1; [Estimation Theory](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb) S3. &nbsp; **Feeds into:** Session 3 (kernel adaptive filters).

---

## 3. Distributions Over Functions

💡 **Intuition.** A GP is [Bayesian estimation](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb) upgraded from parameters to *whole functions*: the kernel plays the prior ('smooth functions with lengthscale ℓ are likely'), data updates it, and the posterior at any test point is a Gaussian — mean **and variance**, in closed form. The error bars behave the way honesty demands: pinched near data, ballooning in the gaps. Where a neural net extrapolates with confidence it hasn't earned, a GP *tells you it doesn't know*.

In [ ]:
# calibration audit on fresh data

# YOUR CODE HERE


**What just happened.** Two things — a posterior with error bars, and a **test of whether those error bars are honest**. The band pinches where data is dense and widens in the gaps, exactly as the formula demands. And on 500 fresh points, **92%** fall inside the ±2σ band against a nominal 95%.

**The shape of the band is not a stylistic choice; it falls out of the algebra.** The posterior variance is $k(x_*,x_*) - k_*^TK^{-1}k_*$ — prior uncertainty *minus* what the data explains. Near an observation the subtracted term is large and the band collapses toward the noise level; far from any observation it vanishes and the variance returns to the prior. **Nothing was tuned to make the picture look right.**

**Note also what the posterior mean is.** It is *identical* to the kernel ridge fit from Session 1, with $\lambda = \sigma_n^2$. Same solve, same numbers. The GP's entire addition is the second moment — which is why it costs nothing extra to have.

**Now the audit, which is the part most GP tutorials omit.** A pretty ±2σ band proves nothing; the claim it makes is falsifiable, so falsify it. Draw fresh data, count the fraction inside the band, compare against 95%. **This turns an uncertainty claim into a measurement**, and it is the habit worth taking from this cell.

**And the result is 92%, which is undercoverage — the band is slightly too narrow, and the reason is diagnosable.** Three percentage points on 500 points is about 15 examples, against a binomial standard error of $\sqrt{0.95 \times 0.05/500} \approx 1\%$, so this is roughly **three standard errors low**: a real effect, not sampling noise. The cause is the target's `0.5*sign(x)` discontinuity. The RBF kernel encodes a prior that functions are infinitely differentiable, so near $x = 0$ the GP sees data on both sides, assumes a smooth transition, and reports *high confidence* — while the truth jumps by 1.0. Residuals near the kink are far larger than the model believes.

**So the audit did its job: it detected a misspecified prior.** That is a better outcome than a demo returning exactly 95%. **Calibration is a property of the model–data pair, not of Bayesian machinery in the abstract** — a GP gives you *coherent* uncertainty given its assumptions, and coherent is not the same as correct. Get the kernel wrong and you get confidently wrong error bars, with all the elegance intact.

**Three ways to fix it, each a real practice.** Switch to a Matérn-1/2 kernel, whose sample paths are non-differentiable and which handles kinks comfortably. Learn $\ell$ and $\sigma_n$ by maximising the marginal likelihood instead of fixing them by hand — the principled version of Session 1's lengthscale question. Or accept the misspecification and inflate $\sigma_n$ to buy coverage, which is honest about the width but not about the cause.

**Still, keep the comparison that motivates the session in view.** The [ANN workshop](./Intro_ANN/Intro_ANN.ipynb) boundary plot showed a network confidently colouring regions it had never seen. This GP is imperfectly calibrated at 92% and **widens its band in every gap**, which is a categorically different failure from unearned confidence. When data is scarce and a confident wrong answer is expensive, that difference is worth more than accuracy.

---
### 🕐 Session 3 of 3 — *Kernel Adaptive Filters: KLMS* (~40 min)
**Goal:** run LMS in the RKHS: a nonlinear adaptive filter, one sample at a time.
**Builds on:** Session 1; [APA workshop](../Intro_Time_Series/Intro_AdFilt_APA.ipynb).

---

## 4. LMS Meets the Kernel

💡 **Intuition.** [LMS](../Intro_Time_Series/Intro_AdFilt_APA.ipynb) updates a weight vector; **KLMS** runs the identical update *in the RKHS* — by the representer theorem the filter is a growing sum of kernel bumps, one planted on each sample, weighted by $\mu e_n$: $f_n = f_{n-1} + \mu e_n \, k(x_n, \cdot)$. It learns *nonlinear* systems online with LMS's simplicity. The price is the growing dictionary — practical variants (QKLMS) merge nearby bumps to cap it.

In [ ]:
# nonlinear system id: y = tanh of a filtered input — LMS can't, KLMS can

# YOUR CODE HERE


**What just happened.** Steady-state MSE **0.0931 for linear LMS against 0.0288 for KLMS** — a factor of 3.2, or **5.1 dB**. The learning curves show the mechanism: linear LMS descends and then **flattens onto a floor**, while KLMS keeps going.

**The linear filter's floor is a representational limit, not a convergence one — and that distinction is the whole session.** The system is $\tanh(1.5 \times \mathrm{FIR}(u))$: a linear filter followed by a saturating nonlinearity. No linear filter can produce a saturating output, at any tap count, with any step size, given any amount of data. **More taps buy nothing.** The residual is the part of the nonlinearity the model cannot express, and the curve flattening is that impossibility made visible.

**KLMS clears the floor because its hypothesis class contains the target.** The update $f_n = f_{n-1} + \mu e_n k(x_n, \cdot)$ is LMS written in the RKHS, and unrolling it from $f_0 = 0$ gives $f_n = \sum_i \mu e_i k(x_i, \cdot)$ — the representer theorem appearing **for free**, as a consequence of the update rather than as a theorem invoked. Each sample plants a Gaussian bump at its own location with height proportional to the error it caused. The filter is a landscape of bumps built wherever it was surprised.

**Now the number the plot title does not mention: KLMS has not converged either.** The additive noise is $0.05\sigma$, so the achievable MSE floor is $0.05^2 = 0.0025$. KLMS sits at 0.0288 — **11× above it**. So the honest reading is that the kernel buys what no linear width can, *and* that 2000 samples with $\mu = 0.5$, $\ell = 1$ is not enough to finish the job. Both facts live in the same two numbers.

**And the cost is the counterweight this session exists to deliver.** The dictionary grows by **one centre per sample, forever** — 2000 centres by the end, none pruned. Prediction at step $t$ costs $O(t)$, so the run is $O(N^2)$ in time and $O(N)$ in memory, both unbounded. Look at the inner loop: `np.array(centers)` rebuilds the entire dictionary on every single sample, which is why this is a teaching implementation and not a deployable filter. **An adaptive filter whose cost grows with uptime cannot run in a real-time loop.**

**Which is exactly what the practical variants attack.** QKLMS quantises the input space and *merges* a new sample into an existing centre when it falls within a threshold, capping the dictionary at a size set by the input distribution rather than by the sample count. Novelty criteria and coherence-based sparsification do the same job by different tests. All of them answer one question: **which of these bumps do I actually need?**

**Zoom out and the trade is the story of the field.** Kernel methods are exact, uncertainty-aware, and **grow with the data they have seen**. Neural networks are approximate and fixed-size. When data is scarce and error bars matter, the first wins — as Session 2 showed. When data is enormous, the dictionary problem is fatal and the second wins. Understanding *why* networks won is worth as much as knowing that they did.

## 5. Conclusion

Kernels buy nonlinearity at linear-algebra prices; GPs add honest uncertainty; KLMS carries it all online. When data is scarce and error bars matter, this toolbox still beats deep learning — and when data is huge, you'll understand *why* networks won (the dictionary problem).

---
## Where next

- [Uncertainty in ML](./Uncertainty_in_ML.ipynb) — chasing GP-quality error bars with deep models.
- [RLS workshop](../Intro_Time_Series/Intro_RLS.ipynb) — KRLS: the recursive version.
- [Hilbert Spaces](../Intro_Math/Hilbert_Spaces/Hilbert_Spaces.ipynb) — the geometry under all of it.